# 03 — CA Knowledge Copilot: Semantic Retrieval & Grounded RAG Benchmarking

**Objective:**
Build and benchmark an authoritative, zero-hallucination Information Retrieval and Retrieval-Augmented Generation (RAG) engine for Chartered Accountants, grounded strictly in official ICAI pronouncements, the Companies Act 2013, Income Tax Act 1961, CGST Act 2017, and Indian Accounting Standards (Ind AS).

**Decisions Supported:**
- Authoritative regulatory query answering with verifiable statutory section citations.
- Reliable abstention when queried on topics outside the verified regulatory corpus (no hallucinated guidance).
- Deterministic sub-millisecond document retrieval for the CA Career OS Copilot.


## 1. Research Questions & Hypotheses

- **H1:** Keyword overlap (lexical matching) fails on natural phrasing where professional terminology uses synonyms (e.g., "cooling-off period" vs "tenure restriction").
- **H2:** A sublinear TF-IDF vector retrieval engine with n-grams will achieve $Recall@1 \ge 0.85$ and $Recall@3 = 1.00$ on curated CA benchmark queries.
- **H3:** Out-of-corpus adversarial queries (e.g., non-existent standards, speculative crypto advice) yield similarity $< 0.15$, enabling an unambiguous `ABSTAIN` gate that eliminates hallucinations.


In [1]:
import os
import sys
import json
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
np.random.seed(SEED)

print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | Scikit-Learn: {sklearn.__version__}")


NumPy: 2.4.6 | Pandas: 3.0.3 | Scikit-Learn: 1.9.0


In [2]:
CORPUS_PATH = Path("/home/vishal/Downloads/CA Carrer OS/data/external/icai_regulatory_corpus.json")

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus_data = json.load(f)

docs = corpus_data['documents']
queries = corpus_data['benchmark_queries']

print(f"Loaded {len(docs)} official statutory documents.")
print(f"Loaded {len(queries)} expert-curated benchmark queries.")

docs_df = pd.DataFrame(docs)
docs_df[['id', 'statute', 'section', 'title', 'category']]


Loaded 8 official statutory documents.
Loaded 8 expert-curated benchmark queries.


,id,statute,section,title,category
0,doc-coact-139,"Companies Act, 2013",Section 139,Appointment of Statutory Auditors & Mandatory ...,CORPORATE_LAW_AUDIT
1,doc-coact-143,"Companies Act, 2013",Section 143(12),Reporting on Fraud by Statutory Auditor to Cen...,CORPORATE_LAW_AUDIT
2,doc-caro-2020,"Companies (Auditor's Report) Order, 2020",CARO 2020 Order Clause 3,Mandatory Inclusions in CARO 2020 Audit Report,AUDITING_STANDARDS
3,doc-indas-115,Indian Accounting Standards (Ind AS),Ind AS 115,Revenue from Contracts with Customers — 5-Step...,ACCOUNTING_STANDARDS
4,doc-indas-116,Indian Accounting Standards (Ind AS),Ind AS 116,Leases — Right-of-Use (ROU) Asset and Lease Li...,ACCOUNTING_STANDARDS
5,doc-itact-44ab,"Income Tax Act, 1961",Section 44AB,Mandatory Tax Audit Thresholds and Form 3CD Fi...,DIRECT_TAX
6,doc-itact-92c,"Income Tax Act, 1961",Section 92C,Computation of Arm's Length Price in Internati...,TRANSFER_PRICING
7,doc-icai-ethics-1,ICAI Code of Ethics (12th Edition 2020),"First Schedule, Part I, Clause 6",Solicitation of Professional Work & Advertisin...,ICAI_PROFESSIONAL_ETHICS


In [3]:
# Lexical baseline: Count of overlapping tokens between query and document text
def tokenize(text):
    return set(re.findall(r'\b[a-z0-9]+\b', text.lower()))

lexical_r1 = 0
lexical_r3 = 0
reciprocal_ranks = []

for q in queries:
    q_tokens = tokenize(q['query'])
    scores = []
    for doc in docs:
        doc_tokens = tokenize(f"{doc['title']} {doc['text']}")
        overlap = len(q_tokens.intersection(doc_tokens))
        scores.append((doc['id'], overlap))
    
    # Sort descending by token overlap
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    ranked_ids = [r[0] for r in ranked]
    expected_ids = set(q['expected_doc_ids'])
    
    # Check Recall@1
    if ranked_ids[0] in expected_ids:
        lexical_r1 += 1
        reciprocal_ranks.append(1.0)
    elif len(ranked_ids) > 1 and ranked_ids[1] in expected_ids:
        reciprocal_ranks.append(1.0 / 2.0)
    elif len(ranked_ids) > 2 and ranked_ids[2] in expected_ids:
        reciprocal_ranks.append(1.0 / 3.0)
    else:
        reciprocal_ranks.append(0.0)
        
    # Check Recall@3
    if any(rid in expected_ids for rid in ranked_ids[:3]):
        lexical_r3 += 1

n_queries = len(queries)
print("=== LEXICAL TOKEN OVERLAP BASELINE ===")
print(f"Recall@1: {lexical_r1 / n_queries:.3f}")
print(f"Recall@3: {lexical_r3 / n_queries:.3f}")
print(f"MRR:      {np.mean(reciprocal_ranks):.3f}")


=== LEXICAL TOKEN OVERLAP BASELINE ===
Recall@1: 0.875
Recall@3: 1.000
MRR:      0.938


In [4]:
# Build sublinear TF-IDF retrieval index with unigrams, bigrams, and statutory section preservation
def build_corpus_text(doc):
    # Header weight amplification for statute, section, and title
    header = f"{doc['statute']} {doc['section']} {doc['title']} {doc['category']}"
    return f"{header} {header} {doc['text']}"

corpus_texts = [build_corpus_text(d) for d in docs]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
    token_pattern=r'(?u)\b\w+\b'
)

doc_vectors = vectorizer.fit_transform(corpus_texts)
print(f"Constructed TF-IDF Vector Index with {doc_vectors.shape[1]} vocabulary dimensions.")

# Evaluate semantic retrieval
semantic_r1 = 0
semantic_r3 = 0
semantic_mrr = []
retrieval_records = []

for q in queries:
    q_vec = vectorizer.transform([q['query']])
    sims = cosine_similarity(q_vec, doc_vectors)[0]
    
    ranked_indices = np.argsort(sims)[::-1]
    top_doc_ids = [docs[idx]['id'] for idx in ranked_indices]
    expected_ids = set(q['expected_doc_ids'])
    
    top_1_hit = top_doc_ids[0] in expected_ids
    top_3_hit = any(did in expected_ids for did in top_doc_ids[:3])
    
    if top_1_hit:
        semantic_r1 += 1
        semantic_mrr.append(1.0)
    elif top_doc_ids[1] in expected_ids:
        semantic_mrr.append(0.5)
    elif top_doc_ids[2] in expected_ids:
        semantic_mrr.append(1.0 / 3.0)
    else:
        semantic_mrr.append(0.0)
        
    if top_3_hit:
        semantic_r3 += 1
        
    retrieval_records.append({
        'query_id': q['query_id'],
        'query': q['query'],
        'top_retrieved_id': top_doc_ids[0],
        'top_score': float(sims[ranked_indices[0]]),
        'expected_id': q['expected_doc_ids'][0],
        'hit@1': top_1_hit,
        'hit@3': top_3_hit
    })

print("=== TF-IDF VECTOR RETRIEVAL PERFORMANCE ===")
print(f"Recall@1: {semantic_r1 / n_queries:.3f}")
print(f"Recall@3: {semantic_r3 / n_queries:.3f}")
print(f"MRR:      {np.mean(semantic_mrr):.3f}")

retrieval_df = pd.DataFrame(retrieval_records)
retrieval_df[['query_id', 'top_retrieved_id', 'top_score', 'hit@1', 'hit@3']]


Constructed TF-IDF Vector Index with 1033 vocabulary dimensions.
=== TF-IDF VECTOR RETRIEVAL PERFORMANCE ===
Recall@1: 1.000
Recall@3: 1.000
MRR:      1.000


,query_id,top_retrieved_id,top_score,hit@1,hit@3
0,q-01,doc-coact-139,0.382815,True,True
1,q-02,doc-coact-143,0.431482,True,True
2,q-03,doc-indas-115,0.320247,True,True
3,q-04,doc-indas-116,0.278273,True,True
4,q-05,doc-itact-44ab,0.350844,True,True
5,q-06,doc-itact-92c,0.288294,True,True
6,q-07,doc-icai-ethics-1,0.236697,True,True
7,q-08,doc-caro-2020,0.302891,True,True


In [5]:
# Verify that the retrieved documents provide exact expected citations
citation_hits = 0

for q in queries:
    q_vec = vectorizer.transform([q['query']])
    sims = cosine_similarity(q_vec, doc_vectors)[0]
    top_idx = np.argmax(sims)
    top_doc = docs[top_idx]
    
    # Check if expected citations appear in the retrieved document
    expected_citations = q['expected_citations']
    doc_full_text = f"{top_doc['statute']} {top_doc['section']} {top_doc['title']} {top_doc['text']}".lower()
    matched_citations = [c for c in expected_citations if c.lower() in doc_full_text]
    
    if len(matched_citations) == len(expected_citations):
        citation_hits += 1

citation_precision = citation_hits / len(queries)
print(f"Citation Grounding Precision: {citation_precision * 100:.1f}%")
assert citation_precision >= 0.85, "Citation grounding below required quality threshold"


Citation Grounding Precision: 100.0%


In [6]:
# Test out-of-corpus and ungrounded adversarial queries
adversarial_queries = [
    "What is the capital gains tax on unauthorized cryptocurrency mining in Dubai?",
    "Explain the guidelines under Ind AS 999 for quantum accounting reserves.",
    "Can you give me guaranteed stock trading tips for Nifty 50 call options?"
]

ABSTENTION_THRESHOLD = 0.15

print("=== ABSTENTION GATE ADVERSARIAL EVALUATION ===")
for adv_q in adversarial_queries:
    q_vec = vectorizer.transform([adv_q])
    sims = cosine_similarity(q_vec, doc_vectors)[0]
    top_score = float(np.max(sims))
    
    if top_score < ABSTENTION_THRESHOLD:
        status = "ABSTAIN (Safe: Zero hallucination triggered)"
    else:
        status = f"FLAG: Score {top_score:.3f} exceeded threshold"
        
    print(f"Query: '{adv_q[:60]}...'")
    print(f"Max Similarity Score: {top_score:.4f} -> {status}\n")
    assert top_score < ABSTENTION_THRESHOLD, f"Abstention failed for: {adv_q}"

print("✅ All adversarial out-of-corpus queries safely rejected by Abstention Gate!")


=== ABSTENTION GATE ADVERSARIAL EVALUATION ===
Query: 'What is the capital gains tax on unauthorized cryptocurrency...'
Max Similarity Score: 0.1160 -> ABSTAIN (Safe: Zero hallucination triggered)

Query: 'Explain the guidelines under Ind AS 999 for quantum accounti...'
Max Similarity Score: 0.1479 -> ABSTAIN (Safe: Zero hallucination triggered)

Query: 'Can you give me guaranteed stock trading tips for Nifty 50 c...'
Max Similarity Score: 0.1087 -> ABSTAIN (Safe: Zero hallucination triggered)

✅ All adversarial out-of-corpus queries safely rejected by Abstention Gate!


In [7]:
vector_store_path = Path("/home/vishal/Downloads/CA Carrer OS/models/regulatory_vector_store.pkl")
vector_meta_path = Path("/home/vishal/Downloads/CA Carrer OS/models/regulatory_vector_store_metadata.json")

vector_store_payload = {
    "vectorizer": vectorizer,
    "doc_vectors": doc_vectors,
    "documents": docs,
    "abstention_threshold": ABSTENTION_THRESHOLD,
    "version": "1.0.0",
    "metrics": {
        "recall_at_1": float(semantic_r1 / n_queries),
        "recall_at_3": float(semantic_r3 / n_queries),
        "mrr": float(np.mean(semantic_mrr)),
        "citation_precision": float(citation_precision)
    }
}

with open(vector_store_path, "wb") as f:
    pickle.dump(vector_store_payload, f)

with open(vector_meta_path, "w", encoding="utf-8") as f:
    json.dump({
        "store_name": "ICAI_Regulatory_Vector_Store",
        "doc_count": len(docs),
        "vocabulary_size": int(doc_vectors.shape[1]),
        "abstention_threshold": ABSTENTION_THRESHOLD,
        "metrics": {
            "recall_at_1": float(semantic_r1 / n_queries),
            "recall_at_3": float(semantic_r3 / n_queries),
            "mrr": float(np.mean(semantic_mrr)),
            "citation_precision": float(citation_precision)
        },
        "artifact_path": str(vector_store_path)
    }, f, indent=2)

print(f"✅ Serialized Regulatory Vector Store to {vector_store_path.name}")


✅ Serialized Regulatory Vector Store to regulatory_vector_store.pkl
